In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, glob

cands = glob.glob('/content/drive/MyDrive/train') + glob.glob('/content/drive/MyDrive/*/train')
print("المجلدات الموجودة:", cands)

train_dir = cands[0]
classes = sorted(d for d in os.listdir(train_dir)
                 if os.path.isdir(f'{train_dir}/{d}'))
counts = {c: len(os.listdir(f'{train_dir}/{c}')) for c in classes}

print(len(classes), "فئة |", sum(counts.values()), "صورة")
for c, n in sorted(counts.items(), key=lambda kv: kv[1]):
    print(f"{n:5d}  {c}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
المجلدات الموجودة: []


IndexError: list index out of range

In [ ]:
!find /content/drive/MyDrive -maxdepth 5 -type d -iname "*train*"

/content/drive/MyDrive/food_project/MFOOD/train


In [ ]:
import os

train_dir = '/content/drive/MyDrive/food_project/MFOOD/train'

classes = sorted(d for d in os.listdir(train_dir)
                 if os.path.isdir(f'{train_dir}/{d}'))
counts = {c: len(os.listdir(f'{train_dir}/{c}')) for c in classes}

print(len(classes), "فئة |", sum(counts.values()), "صورة")
for c, n in sorted(counts.items(), key=lambda kv: kv[1]):
    print(f"{n:5d}  {c}")

70 فئة | 8602 صورة
   26  traditional macaroon
   37  tomatoes and onion salad
   41  TAKTOUKA
   43  zitoun
   45  sweet bread
   45  tkalya
   46  sellou
   48  amlou
   53  sfenje
   56  snowballs
   57  traditional bread
   57  white beans with tomatoes
   61  tagine with quince
   61  tagine with vegetables
   69  tagine with artichokes and peas
   84  seffa with rice
   86  crackers with almonds (ghriba)
   95  nougat
  103  fekkas
  104  better beldi
  104  chebakia
  104  gazelle horn
  105  BISSARA(feves puree)
  105  maakouda
  110  FEVES with sauce
  110  chicken with potatoes and olives
  117  bahla
  118  Feet of beef
  118  lentils
  125  briouate  with almonds
  141  meatball with tomato sauce
  142  salmon
  143  jam
  145  basbousa
  150  apple
  150  banana
  150  batbout
  150  beghrir
  150  caesar salad
  150  chicken basstila
  150  chicken nuggets
  150  chocolate cake
  150  couscous
  150  croissant
  150  dates
  150  fish and vegetables
  150  fish basstila
 

In [ ]:
!pip -q install imagehash

import os, random, shutil
from PIL import Image
import imagehash
from google.colab import drive
drive.mount('/content/drive')

SRC = '/content/drive/MyDrive/food_project/MFOOD/train'     # نفس مسار مجلد train الذي استعملته في الخلية السابقة
CLEAN = '/content/food_clean'
OUT = '/content/food'
THRESH = 4    # أقصى فرق بين بصمتي صورتين لنعتبرهما شبه متطابقتين

for d in (CLEAN, OUT):
    shutil.rmtree(d, ignore_errors=True)

classes = sorted(d for d in os.listdir(SRC) if os.path.isdir(f'{SRC}/{d}'))
random.seed(42)

bad = 0
dup_imgs = 0
split_counts = {'train': 0, 'val': 0, 'test': 0}

for cls in classes:
    os.makedirs(f'{CLEAN}/{cls}', exist_ok=True)
    items = []   # (المسار، البصمة)
    for i, f in enumerate(sorted(os.listdir(f'{SRC}/{cls}'))):
        try:
            with Image.open(f'{SRC}/{cls}/{f}') as img:
                img = img.convert('RGB')
                h = imagehash.phash(img)
                p = f'{CLEAN}/{cls}/{i:04d}.jpg'
                img.save(p, 'JPEG', quality=95)
            items.append((p, h))
        except Exception:
            bad += 1

    # تجميع الصور شبه المتطابقة
    n = len(items)
    parent = list(range(n))

    def find(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]
            a = parent[a]
        return a

    for a in range(n):
        for b in range(a + 1, n):
            if items[a][1] - items[b][1] <= THRESH:
                parent[find(a)] = find(b)

    groups = {}
    for idx in range(n):
        groups.setdefault(find(idx), []).append(items[idx][0])
    groups = list(groups.values())
    dup_imgs += sum(len(g) for g in groups if len(g) > 1)
    random.shuffle(groups)

    # توزيع المجموعات
    n_test = max(1, round(0.15 * n))
    n_val = max(1, round(0.15 * n))
    assign = {'test': [], 'val': [], 'train': []}
    for g in groups:
        if len(assign['test']) < n_test:
            assign['test'] += g
        elif len(assign['val']) < n_val:
            assign['val'] += g
        else:
            assign['train'] += g

    for split, paths in assign.items():
        os.makedirs(f'{OUT}/{split}/{cls}', exist_ok=True)
        for p in paths:
            shutil.copy(p, f'{OUT}/{split}/{cls}/{os.path.basename(p)}')
        split_counts[split] += len(paths)

print("ملفات تالفة تم تخطيها:", bad)
print("صور لها نسخة شبه مطابقة:", dup_imgs)
print("التقسيم:", split_counts)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


ملفات تالفة تم تخطيها: 3
صور لها نسخة شبه مطابقة: 2344
التقسيم: {'train': 6012, 'val': 1296, 'test': 1291}


In [ ]:
import os, shutil
base = '/content/food'

shutil.make_archive('/content/drive/MyDrive/food_project/food_split', 'zip', base)
print("تم حفظ النسخة الاحتياطية في Drive")

classes = sorted(os.listdir(f'{base}/train'))
rows = [(c, *[len(os.listdir(f'{base}/{s}/{c}')) for s in ('train', 'val', 'test')])
        for c in classes]

print("\nفئات ينقصها جزء (تدريب، تحقق، اختبار):")
for r in rows:
    if min(r[1:]) == 0:
        print(r)

print("\nأقل 8 فئات في التدريب (الفئة، تدريب، تحقق، اختبار):")
for r in sorted(rows, key=lambda r: r[1])[:8]:
    print(r)

تم حفظ النسخة الاحتياطية في Drive

فئات ينقصها جزء (تدريب، تحقق، اختبار):

أقل 8 فئات في التدريب (الفئة، تدريب، تحقق، اختبار):
('traditional macaroon', 17, 5, 4)
('tomatoes and onion salad', 25, 6, 6)
('TAKTOUKA', 29, 6, 6)
('sellou', 29, 9, 8)
('tkalya', 30, 8, 7)
('sweet bread', 31, 7, 7)
('zitoun', 31, 6, 6)
('amlou', 34, 7, 7)


In [ ]:
import json
import numpy as np
import tensorflow as tf
from tensorflow import keras

print("GPU:", tf.config.list_physical_devices('GPU'))
IMG, BATCH = (224, 224), 32

def load(split, shuffle):
    return keras.utils.image_dataset_from_directory(
        f'{base}/{split}', image_size=IMG, batch_size=BATCH,
        label_mode='int', shuffle=shuffle, seed=42)

train_raw, val_raw, test_raw = load('train', True), load('val', False), load('test', False)
class_names = train_raw.class_names
NUM = len(class_names)
train_ds = train_raw.prefetch(tf.data.AUTOTUNE)
val_ds = val_raw.prefetch(tf.data.AUTOTUNE)
test_ds = test_raw.prefetch(tf.data.AUTOTUNE)

counts = [max(1, len(os.listdir(f'{base}/train/{c}'))) for c in class_names]
total = sum(counts)
class_weight = {i: total / (NUM * n) for i, n in enumerate(counts)}

base_model = keras.applications.MobileNetV2(
    input_shape=IMG + (3,), include_top=False, weights='imagenet')
base_model.trainable = False

augment = keras.Sequential([
    keras.layers.RandomFlip('horizontal'),
    keras.layers.RandomRotation(0.1),
    keras.layers.RandomZoom(0.1),
])
inputs = keras.Input(shape=IMG + (3,))
x = augment(inputs)
x = keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = keras.layers.GlobalAveragePooling2D()(x)
x = keras.layers.Dropout(0.3)(x)
outputs = keras.layers.Dense(NUM, activation='softmax')(x)
model = keras.Model(inputs, outputs)

metrics = ['accuracy', keras.metrics.SparseTopKCategoricalAccuracy(k=3, name='top3')]
stop = keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=3,
                                     restore_best_weights=True)

# المرحلة 1: ندرّب الطبقة الأخيرة فقط
model.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss='sparse_categorical_crossentropy', metrics=metrics)
model.fit(train_ds, validation_data=val_ds, epochs=10,
          class_weight=class_weight, callbacks=[stop])

# المرحلة 2: Fine-tuning لآخر الطبقات
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False
model.compile(optimizer=keras.optimizers.Adam(1e-5),
              loss='sparse_categorical_crossentropy', metrics=metrics)
model.fit(train_ds, validation_data=val_ds, epochs=10,
          class_weight=class_weight, callbacks=[stop])

P = '/content/drive/MyDrive/food_project'
model.save(f'{P}/food_model_v1.keras')
with open(f'{P}/food_classes.json', 'w') as f:
    json.dump(class_names, f, ensure_ascii=False)
print("تم حفظ النموذج وأسماء الفئات")

GPU: []
Found 6012 files belonging to 70 classes.
Found 1296 files belonging to 70 classes.
Found 1291 files belonging to 70 classes.
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/10
188/188 ━━━━━━━━━━━━━━━━━━━━ 397s 2s/step - accuracy: 0.4421 - loss: 2.3889 - top3: 0.6264 - val_accuracy: 0.7122 - val_loss: 1.1590 - val_top3: 0.8665
Epoch 2/10
188/188 ━━━━━━━━━━━━━━━━━━━━ 400s 2s/step - accuracy: 0.7269 - loss: 1.0360 - top3: 0.8894 - val_accuracy: 0.7593 - val_loss: 0.8643 - val_top3: 0.9066
Epoch 3/10
188/188 ━━━━━━━━━━━━━━━━━━━━ 402s 2s/step - accuracy: 0.7967 - loss: 0.7396 - top3: 0.9311 - val_accuracy: 0.7878 - val_loss: 0.7485 - val_top3: 0.9259
Epoch 4/10
188/188 ━━━━━━━━━━━━━━━━━━━━ 391s 2s/step - accuracy: 0.8282 - loss: 0.6131 - top3: 0.9461 - val_accuracy: 0.7978 - val_loss: 0.7066 - val_top3: 0.9252
Epoch 5/10
188/188 ━━━━━━━━━━━━━━━━━━━━ 370s 2s/step - accuracy: 0.8485 - loss: 0.5149 - top3: 0.9597 - val_accuracy: 0.8094 - val_loss: 0.6915 - val_top3: 0.9159
Ep

In [ ]:
from sklearn.metrics import confusion_matrix

y_true = np.concatenate([y.numpy() for _, y in test_ds])
probs = model.predict(test_ds, verbose=0)
pred = probs.argmax(axis=1)

acc = (pred == y_true).mean()
top3 = np.mean([t in np.argsort(p)[-3:] for t, p in zip(y_true, probs)])
print(f"دقة الاختبار: {acc:.4f} | الطبق الصحيح ضمن أعلى 3 توقعات: {top3:.4f}")

cm = confusion_matrix(y_true, pred, labels=range(NUM))
n_test = cm.sum(axis=1)
recall = cm.diagonal() / np.maximum(n_test, 1)

print("\nأضعف 10 فئات (recall، عدد صور الاختبار):")
for i in np.argsort(recall)[:10]:
    print(f"{recall[i]:.2f}  ({n_test[i]:3d})  {class_names[i]}")

print("\nأكثر الأخطاء تكراراً (الحقيقي ← توقع النموذج):")
off = cm.copy()
np.fill_diagonal(off, 0)
for flat in np.argsort(off, axis=None)[::-1][:10]:
    i, j = divmod(flat, NUM)
    print(f"{off[i, j]:2d}  {class_names[i]}  ←  {class_names[j]}")

دقة الاختبار: 0.8249 | الطبق الصحيح ضمن أعلى 3 توقعات: 0.9357

أضعف 10 فئات (recall، عدد صور الاختبار):
0.31  ( 16)  FEVES with sauce
0.33  (  6)  zitoun
0.44  (  9)  white beans with tomatoes
0.50  ( 22)  meatball with tomato sauce
0.50  ( 12)  tagine with artichokes and peas
0.54  ( 13)  seffa with rice
0.56  ( 18)  Feet of beef
0.59  ( 41)  liver with sauce
0.59  ( 17)  chicken with potatoes and olives
0.62  (  8)  sellou

أكثر الأخطاء تكراراً (الحقيقي ← توقع النموذج):
 6  liver with sauce  ←  meatball with tomato sauce
 5  meatball with tomato sauce  ←  liver with sauce
 4  seffa with rice  ←  seffa
 4  chicken with potatoes and olives  ←  tagine with quince
 4  Feet of beef  ←  white beans with tomatoes
 3  traditional bread  ←  snowballs
 3  FEVES with sauce  ←  Feet of beef
 3  tagine with artichokes and peas  ←  chicken with potatoes and olives
 3  liver with sauce  ←  zaalouk
 3  basbousa  ←  kaak


In [ ]:
import json
import gradio as gr
import numpy as np
from PIL import Image
from tensorflow import keras
from google.colab import drive

drive.mount('/content/drive')
P = '/content/drive/MyDrive/food_project'

model = keras.models.load_model(f'{P}/food_model_v1.keras')
with open(f'{P}/food_classes.json') as f:
    class_names = json.load(f)
print("تم تحميل النموذج:", len(class_names), "فئة")

def predict(img):
    if img is None:
        return None
    img = img.convert('RGB').resize((224, 224))
    x = np.expand_dims(np.array(img, dtype='float32'), axis=0)
    probs = model.predict(x, verbose=0)[0]
    return {class_names[i]: float(probs[i]) for i in range(len(class_names))}

HERO = """
<div style="background:linear-gradient(135deg,#9a3412,#ea580c 55%,#fdba74);
            border-radius:20px;padding:30px 20px;text-align:center;color:white;
            margin-bottom:14px;">
  <h1 style="margin:0 0 6px 0;font-size:1.9rem;color:white;">🍽️ ما هذا الطبق؟</h1>
  <p style="margin:0;font-size:1.05rem;color:white;opacity:.95;">
     صوّر طبقاً بالهاتف أو ارفع صورته، وسأخمّن اسمه</p>
</div>
"""

NOTE = """
<div style="text-align:center;font-size:.9rem;opacity:.85;margin-top:12px;">
  ⚠️ مشروع تعليمي: النموذج يعرف 70 طبقاً فقط، فإن أريته شيئاً آخر
  (أو شيئاً ليس طعاماً) فسيختار أقرب طبق من الـ 70، وقد يخطئ.
  أفضل النتائج مع صورة واضحة لطبق واحد من الأعلى.
</div>
"""

with gr.Blocks(title="ما هذا الطبق؟") as demo:
    gr.HTML(HERO)
    inp = gr.Image(type="pil", sources=["upload", "webcam"],
                   label="صورة الطبق (ارفع صورة أو التقطها)", height=360)
    btn = gr.Button("🔍 ما اسم هذا الطبق؟", variant="primary")
    out = gr.Label(num_top_classes=3, label="أعلى 3 توقعات")
    gr.HTML(NOTE)
    btn.click(predict, inputs=inp, outputs=out)

demo.launch(share=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
تم تحميل النموذج: 70 فئة
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c1edc9314b1b6b9ec9.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [1]:
import json
import gradio as gr
import numpy as np
from PIL import Image
from tensorflow import keras
from google.colab import drive

drive.mount('/content/drive')
P = '/content/drive/MyDrive/food_project'

model = keras.models.load_model(f'{P}/food_model_v1.keras')
with open(f'{P}/food_classes.json') as f:
    class_names = json.load(f)
print("تم تحميل النموذج:", len(class_names), "فئة")

AR = {
    "traditional macaroon": "ماكرون تقليدي",
    "tomatoes and onion salad": "سلطة الطماطم والبصل",
    "TAKTOUKA": "تكتوكة",
    "zitoun": "زيتون",
    "sweet bread": "خبز حلو",
    "tkalya": "تقلية",
    "sellou": "سلّو",
    "amlou": "أملو",
    "sfenje": "سفنج",
    "snowballs": "كرات الثلج (حلوى جوز الهند)",
    "traditional bread": "خبز تقليدي",
    "white beans with tomatoes": "لوبيا بيضاء بالطماطم",
    "tagine with quince": "طاجين بالسفرجل",
    "tagine with vegetables": "طاجين بالخضار",
    "tagine with artichokes and peas": "طاجين بالخرشوف والبازلاء",
    "seffa with rice": "سفة بالأرز",
    "crackers with almonds (ghriba)": "غريبة باللوز",
    "nougat": "نوغا",
    "fekkas": "فقاص",
    "better beldi": "زبدة بلدية",
    "chebakia": "شباكية",
    "gazelle horn": "كعب الغزال",
    "BISSARA(feves puree)": "بيصارة",
    "maakouda": "معقودة",
    "FEVES with sauce": "فول بالصلصة",
    "chicken with potatoes and olives": "دجاج بالبطاطس والزيتون",
    "bahla": "بهلة",
    "Feet of beef": "كرعين البقر",
    "lentils": "عدس",
    "briouate  with almonds": "بريوات باللوز",
    "briouate with almonds": "بريوات باللوز",
    "meatball with tomato sauce": "كفتة بصلصة الطماطم",
    "salmon": "سلمون",
    "jam": "مربى",
    "basbousa": "بسبوسة",
    "apple": "تفاح",
    "banana": "موز",
    "batbout": "بطبوط",
    "beghrir": "بغرير",
    "caesar salad": "سلطة سيزر",
    "chicken basstila": "بسطيلة بالدجاج",
    "chicken nuggets": "ناجتس الدجاج",
    "chocolate cake": "كعكة الشوكولاتة",
    "couscous": "كسكس",
    "croissant": "كرواسون",
    "dates": "تمر",
    "fish and vegetables": "سمك بالخضار",
    "fish basstila": "بسطيلة بالسمك",
    "french fries": "بطاطس مقلية",
    "fried calamari": "حبّار مقلي",
    "hamburger": "همبرغر",
    "harcha": "حرشة",
    "harira": "حريرة",
    "lasagne": "لازانيا",
    "meat brochettes": "أسياخ لحم مشوي",
    "msamen": "مسمن",
    "omelette": "عجّة بالبيض (أومليت)",
    "orange": "برتقال",
    "paella": "باييا",
    "pear": "إجاص",
    "pizza": "بيتزا",
    "rfissa": "رفيسة",
    "roasted chicken": "دجاج مشوي",
    "seffa": "سفة",
    "spagetti": "سباغيتي",
    "tagine with beef": "طاجين باللحم",
    "zaalouk": "زعلوك",
    "mhancha": "محنشة",
    "rghayf": "رغايف",
    "kaak": "كعك",
    "liver with sauce": "كبدة بالصلصة",
}

names_ar = [AR.get(c, c) for c in class_names]
missing = [c for c in class_names if c not in AR]
print("فئات بلا ترجمة (ستظهر بالاسم الأصلي):", missing)

def predict(img):
    if img is None:
        return None
    img = img.convert('RGB').resize((224, 224))
    x = np.expand_dims(np.array(img, dtype='float32'), axis=0)
    probs = model.predict(x, verbose=0)[0]
    return {names_ar[i]: float(probs[i]) for i in range(len(names_ar))}

HERO = """
<div style="background:linear-gradient(135deg,#9a3412,#ea580c 55%,#fdba74);
            border-radius:20px;padding:30px 20px;text-align:center;color:white;
            margin-bottom:14px;direction:rtl;">
  <h1 style="margin:0 0 6px 0;font-size:1.9rem;color:white;">🍽️ ما هذا الطبق؟</h1>
  <p style="margin:0 0 14px 0;font-size:1.05rem;color:white;opacity:.95;">
     صوّر طبقاً بالهاتف أو ارفع صورته، وسأخمّن اسمه</p>
  <span style="display:inline-block;background:rgba(255,255,255,.22);
               border:1px solid rgba(255,255,255,.55);border-radius:999px;
               padding:6px 18px;font-size:1rem;color:white;">
     ✨ من إنجاز: <b>حمزة اولبياض</b></span>
</div>
"""

FOOT = """
<div style="text-align:center;font-size:.9rem;opacity:.9;margin-top:12px;direction:rtl;">
  <div>⚠️ مشروع تعليمي: النموذج يعرف 70 طبقاً فقط، فإن أريته شيئاً آخر
  (أو شيئاً ليس طعاماً) فسيختار أقرب طبق من الـ 70، وقد يخطئ.
  أفضل النتائج مع صورة واضحة لطبق واحد من الأعلى.</div>
  <div style="margin-top:8px;">تصميم وتدريب النموذج: <b>حمزة اولبياض</b>
  — قاعدة البيانات: MFOOD_70 (رخصة CC BY-SA 4.0)</div>
</div>
"""

with gr.Blocks(title="ما هذا الطبق؟ — حمزة اولبياض") as demo:
    gr.HTML(HERO)
    inp = gr.Image(type="pil", sources=["upload", "webcam"],
                   label="صورة الطبق (ارفع صورة أو التقطها)", height=360)
    btn = gr.Button("🔍 ما اسم هذا الطبق؟", variant="primary")
    out = gr.Label(num_top_classes=3, label="أعلى 3 توقعات")
    gr.HTML(FOOT)
    btn.click(predict, inputs=inp, outputs=out)

demo.launch(share=True)

Mounted at /content/drive
تم تحميل النموذج: 70 فئة
فئات بلا ترجمة (ستظهر بالاسم الأصلي): []
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://36aaa1537a44158a48.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
